### Daten analyse

In [39]:
### TXREG Daten ziehen   ins df Dataframe
 
import requests
import os
import pandas as pd
from io import StringIO
import json

# Define the URL and credentials
url = "https://cloud.h-da.de/public.php/webdav"
username = "DNQpTLHqBQzbkFL"

with open('secret.json', 'r') as f:
    data = json.load(f)
    password = data['IDEN_DATA_SHARE_PW']

# Make the GET request with basic authentication
response = requests.get(url, auth=(username, password), headers={"X-Requested-With": "XMLHttpRequest"})

# Check if the request was successful
if response.status_code == 200:
    # Read the CSV data
    df = pd.read_csv(StringIO(response.text))
    print('Data retrieved successfully')
else:
    print(f"Failed to retrieve data: {response.status_code}")

Data retrieved successfully


In [40]:
# Gruppiere die Spalten nach ihren Datentypen
datatypes = df.dtypes

# Iteriere über jeden einzigartigen Datentyp
for dtype in datatypes.unique():
    print(f"Spalten mit Datentyp {dtype}  ({len(datatypes[datatypes == dtype].index.tolist())} Merkmale):")
    print(datatypes[datatypes == dtype].index.tolist())
    print()  # Leerzeile zur besseren Übersicht

print('Shape: ')
print(df.shape)
n_1 = df.shape[0]

Spalten mit Datentyp float64  (9 Merkmale):
['time', 'donor_age_years', 'donor_height_cm', 'donor_weight_kg', 'donor_creatinin_umol_per_l', 'recipient_age_years', 'recipient_height_cm', 'recipient_weight_kg', 'recipient_dialysis_years']

Spalten mit Datentyp int64  (3 Merkmale):
['event', 'recipient_pra', 'transplant_cold_ischemia_time_min']

Spalten mit Datentyp object  (5 Merkmale):
['donor_sex', 'donor_death_reason', 'recipient_sex', 'recipient_diagnosis', 'destination']

Spalten mit Datentyp bool  (7 Merkmale):
['donor_diabetes', 'donor_hypertension', 'donor_smoking', 'donor_hcv', 'recipient_bloodtransfusion', 'recipient_hcv', 'transplant_abo_compat']

Shape: 
(17016, 24)


In [41]:
# Zeilen entfernen mit time <= 0

df = df[df['time']>0]
print(f'{n_1-df.shape[0]} Zeilen entfernt')
print('\nShape nach time>0: ')
print(df2.shape)

1230 Zeilen entfernt

Shape nach time>0: 
(15772, 19)


In [42]:
### Kategorische Merkmale mit zu vielen Ausprägungen entfernen/binarisieren

# transplant_abo_compat hat 2 ausprägungen, wovon eine nur 3 mal vorkommt --- > daher merkmal entfernen
# recipient_diagnosis   248 ausprägungen  -- merkmal rausnehmen, tipp von prof. dr. jahn
# donor_death_reason    41  ausprägungen  -- merkmal rausnehmen, tipp von prof. dr. jahn
df = df.drop(['recipient_diagnosis', 'donor_death_reason','transplant_abo_compat'], axis=1)

# recipient_pra 0 oder 1 setzen und in bool umwandeln, 0 falls 0 sonst 1, tipp von prof. dr. jahn
df['recipient_pra'] = df['recipient_pra'].apply(lambda x: x != 0)

In [43]:
# Merkmal destination, ausprägungen local dünn besetzt  --> zeilen mit der ausprägung rausnehmen
event_counts = df.groupby('destination')['event'].value_counts().unstack(fill_value=0)
#print(event_counts)
df = df[df['destination'] != 'Local']

### Train Test Split

In [44]:
# Merkmale zum trainieren des Classifiers festlegen
X_columns = df.columns.tolist()
X_columns.remove('time')
X_columns.remove('event')

# Gruppiere die Spalten nach ihren Datentypen
datatypes = df.dtypes

# Iteriere über jeden einzigartigen Datentyp
for dtype in datatypes.unique():
    print(f"Spalten mit Datentyp {dtype}  ({len(datatypes[datatypes == dtype].index.tolist())} Merkmale):")
    print(datatypes[datatypes == dtype].index.tolist())
    print()  # Leerzeile zur besseren Übersicht

print('Shape: ')
print(df.shape)

# teile die daten in trainings- und testdaten auf nach cut bei 3 jahren
tau = 3* 365
test_size = 0.2
seed = 42

from utils import create_train_test_data
df_train, df_test,n_events_after_cut_train,portion_censored_after_cut_train,n_events_after_cut_test , portion_censored_after_cut_test = create_train_test_data(tau=tau, data = df, X_columns=X_columns,  seed=seed, test_size=test_size)

print('Train:')
print(f'Anteil der Zensierten beobachtungen nach dem cut bei tau: {round(portion_censored_after_cut_train*100,2)}%')
print(f'Anteil der Events nach dem cut bei tau: {round(n_events_after_cut_train/df_train.shape[0]*100,2)}%')

print('\nTest:')
print(f'Anteil der Zensierten beobachtungen nach dem cut bei tau: {round(portion_censored_after_cut_test*100,2)}%')
print(f'Anteil der Events nach dem cut bei tau: {round(n_events_after_cut_test/df_test.shape[0]*100,2)}%')

Spalten mit Datentyp float64  (9 Merkmale):
['time', 'donor_age_years', 'donor_height_cm', 'donor_weight_kg', 'donor_creatinin_umol_per_l', 'recipient_age_years', 'recipient_height_cm', 'recipient_weight_kg', 'recipient_dialysis_years']

Spalten mit Datentyp int64  (2 Merkmale):
['event', 'transplant_cold_ischemia_time_min']

Spalten mit Datentyp object  (3 Merkmale):
['donor_sex', 'recipient_sex', 'destination']

Spalten mit Datentyp bool  (7 Merkmale):
['donor_diabetes', 'donor_hypertension', 'donor_smoking', 'donor_hcv', 'recipient_bloodtransfusion', 'recipient_hcv', 'recipient_pra']

Shape: 
(15772, 21)
Train:
Anteil der Zensierten beobachtungen nach dem cut bei tau: 19.51%
Anteil der Events nach dem cut bei tau: 18.94%

Test:
Anteil der Zensierten beobachtungen nach dem cut bei tau: 20.1%
Anteil der Events nach dem cut bei tau: 17.69%


### Hyperparameter optimierung des Bagged decision Tree Classifier

In [ ]:

# import numpy as np
# import itertools
# from utils import DecisionTreeBaggingClassifier, ipc_weighted_mse

# # Dummy-Variablen für Training und Test-Daten erstellen
# df_train_dummy = pd.get_dummies(df_train, drop_first=True)
# df_test_dummy = pd.get_dummies(df_test, drop_first=True)

# # Sicherstellen, dass beide Datasets die gleichen Spalten haben
# df_test_dummy = df_test_dummy.reindex(columns=df_train_dummy.columns)

# # Definiere den Parameterraum
# param_rf_raum = { 
#     "n_estimators": [1000,10_000,1000],
#     "max_depth": np.arange(2, 6),
#     "min_samples_split": np.arange(4, 30,2),
#     "max_features": ["sqrt", "log2"],
# }
# params_rf = {   'n_estimators':1000,                        
#                 'max_depth':5,
#                 'min_samples_split':5,
#                 'max_features': 'log2',
#                 'random_state':  42,
#                 'weighted_bootstrapping': True, }

# # Erstelle eine Liste aller möglichen Parameterkombinationen
# from sklearn.model_selection import ParameterGrid
# param_grid = list(ParameterGrid(param_rf_raum))

# best_score = float('inf')
# best_params = None

# for params in param_grid:
#     # Aktualisiere die Parameter für das aktuelle Modell
#     current_params = params_rf.copy()
#     current_params.update(params)
    
#     # Initialisiere und trainiere das Modell
#     clf = DecisionTreeBaggingClassifier(current_params)
#     clf.fit(
#         X=df_train_dummy.drop(
#             ["time", "event", "weights_ipcw", "survived"], axis=1
#         ).values,
#         y=df_train_dummy["survived"].values,
#         sample_weights=df_train_dummy["weights_ipcw"].values,
#     )
    
#     # Vorhersagen auf den Testdaten
#     _, pred = clf.predict_proba(df_test_dummy.drop(
#                 ["weights_ipcw", "survived", "time", "event"], axis=1
#             ).values)
    
#     # Bewertung des Modells
#     rf_mse_ipcw = ipc_weighted_mse(
#         y_true=df_test_dummy["survived"].values, 
#         y_pred=pred,
#         sample_weight=df_test_dummy["weights_ipcw"].values,
#     )
    
#     # Überprüfe, ob dies die besten Parameter sind
#     if rf_mse_ipcw < best_score:
#         best_score = rf_mse_ipcw
#         best_params = params.copy()
#         print(f"Neues bestes IPCW MSE: {best_score} mit Parametern: {best_params}")

# print(f"Bestes IPCW MSE: {best_score}")
# print(f"Beste Hyperparameter: {best_params}")

### Final Bagged decision Tree Classifier

In [60]:
from utils import DecisionTreeBaggingClassifier
from utils import ipc_weighted_mse


params_rf = {   'n_estimators':1000,                        
                'max_depth':9,
                'min_samples_split':20,
                'max_features': 'log2',
                'random_state':  42,
                'weighted_bootstrapping': True, }

# Dummy-Variablen für Training und Test-Daten erstellen
df_train_dummy = pd.get_dummies(df_train, drop_first=True)
df_test_dummy = pd.get_dummies(df_test, drop_first=True)

# Sicherstellen, dass beide Datasets die gleichen Spalten haben
df_test_dummy = df_test_dummy.reindex(columns=df_train_dummy.columns)

clf = DecisionTreeBaggingClassifier(params_rf)
clf.fit(
    X=df_train_dummy.drop(
        ["time", "event", "weights_ipcw", "survived"], axis=1
    ).values,
    y=df_train_dummy["survived"].values,
    sample_weights=df_train_dummy["weights_ipcw"].values,
)

_ , pred  =clf.predict_proba(df_test_dummy.drop(
            ["weights_ipcw", "survived", "time", "event"], axis=1
        ).values)

# Evaluation auf Testdaten
rf_mse_ipcw = ipc_weighted_mse(
    y_true=df_test_dummy["survived"].values, 
    y_pred=pred,
    sample_weight=df_test_dummy["weights_ipcw"].values,
)

print(f"IPCW MSE: {round(rf_mse_ipcw,4)}")

IPCW MSE: 0.1375


### Patienten finden für Vorhersage 

In [38]:
### Find the observation closest to the average patient ###
import gower

# Exclude target variables if they are not features
target_cols = ['time', 'event']
feature_cols = df.columns.difference(target_cols)

# Step 1: Calculate mean for numerical columns (excluding target variables)
numeric_cols = df[feature_cols].select_dtypes(include=['float64', 'int64']).columns
mean_values = df[numeric_cols].mean()

# Step 2: Calculate mode for categorical columns
categorical_cols = df[feature_cols].select_dtypes(include=['object']).columns
mode_values = df[categorical_cols].mode().iloc[0]

# Step 3: Calculate mode for boolean columns
bool_cols = df[feature_cols].select_dtypes(include=['bool']).columns
bool_mode_values = df[bool_cols].mode().iloc[0]

# Step 4: Combine all results into one Series
average_patient = pd.concat([mean_values, bool_mode_values, mode_values])

# Step 5: Reindex average_patient to match feature columns
average_patient = average_patient[feature_cols]

# Step 6: Create average_patient_df with same columns as df (feature columns)
average_patient_df = pd.DataFrame([average_patient], columns=feature_cols)

# Step 7: Ensure data types match between df and average_patient_df
df_features = df[feature_cols]
average_patient_df = average_patient_df.astype(df_features.dtypes)

# Step 8: Compute Gower distances
# Note: Use only feature columns for distance calculation
distances = gower.gower_matrix(df_features, average_patient_df)

# Extract distances to the average patient
distances_to_average = distances[:, 0]

# Find the index with the minimum distance
min_index = distances_to_average.argmin()

# Retrieve the closest observation (including target variables)
closest_observation = df.iloc[min_index]

print("\nObservation closest to the average patient:")
print(closest_observation)
print(f'\nIndex des mean Patienten {min_index}')



Observation closest to the average patient:
time                                   1826.25
event                                        0
donor_age_years                           57.0
donor_sex                                 male
donor_height_cm                          175.0
donor_weight_kg                           78.0
donor_creatinin_umol_per_l                25.6
donor_diabetes                           False
donor_hypertension                       False
donor_smoking                            False
donor_hcv                                False
recipient_age_years                  52.774812
recipient_sex                             male
recipient_height_cm                      173.0
recipient_weight_kg                       84.0
recipient_bloodtransfusion               False
recipient_dialysis_years              4.884326
recipient_hcv                            False
recipient_pra                            False
transplant_cold_ischemia_time_min          829
destination    

In [ ]:
from utils import create_train_test_data
df_train, df_test,n_events_after_cut_train,portion_censored_after_cut_train,n_events_after_cut_test , portion_censored_after_cut_test = create_train_test_data(tau=tau, data = df, X_columns=X_columns,  seed=seed, test_size=0.000000001)



In [36]:
# pdf noch mitreinnehmen , mse_ipcw das gleiche wie der ipc brier scrore, lukas naych referenz fragen
# lukas fragen wefgen  Datenquelle  txREg dataset

df.iloc[15404,:]

time                                   1826.25
event                                        0
donor_age_years                           57.0
donor_sex                                 male
donor_height_cm                          175.0
donor_weight_kg                           78.0
donor_creatinin_umol_per_l                25.6
donor_diabetes                           False
donor_hypertension                       False
donor_smoking                            False
donor_hcv                                False
recipient_age_years                  52.774812
recipient_sex                             male
recipient_height_cm                      173.0
recipient_weight_kg                       84.0
recipient_bloodtransfusion               False
recipient_dialysis_years              4.884326
recipient_hcv                            False
recipient_pra                            False
transplant_cold_ischemia_time_min          829
destination                           Regional
Name: 16623, 